<a href="https://colab.research.google.com/github/fahimahmedcse06-cmyk/chest-xray-pneumonia-detection/blob/main/chest_xray_pneumonia_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub

import kagglehub
import os

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Dataset path:", path)

base = os.path.join(path, "chest_xray")

for split in ["train", "test", "val"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        folder = os.path.join(base, split, cls)
        print(split, cls, len(os.listdir(folder)))

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Dataset path: /kaggle/input/chest-xray-pneumonia
train NORMAL 1341
train PNEUMONIA 3875
test NORMAL 234
test PNEUMONIA 390
val NORMAL 8
val PNEUMONIA 8


In [3]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
base = os.path.join(path, "chest_xray")
print("base:", base)
print(os.listdir(base))

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
base: /kaggle/input/chest-xray-pneumonia/chest_xray
['chest_xray', '__MACOSX', 'val', 'test', 'train']


In [4]:
IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "train"),
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "train"),
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
    shuffle=False,
)

print("Classes:", train_ds.class_names)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.
Found 5216 files belonging to 2 classes.
Using 1043 files for validation.
Found 624 files belonging to 2 classes.
Classes: ['NORMAL', 'PNEUMONIA']


import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "train"),
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "train"),
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
    shuffle=False,
)

print("Classes:", train_ds.class_names)

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "train"),
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "train"),
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH,
    color_mode="rgb",
    shuffle=False,
)

print("Classes:", train_ds.class_names)

In [ ]:
n_normal = len(os.listdir(os.path.join(base, "train", "NORMAL")))
n_pneumonia = len(os.listdir(os.path.join(base, "train", "PNEUMONIA")))
total = n_normal + n_pneumonia

class_weight = {
    0: total / (2 * n_normal),
    1: total / (2 * n_pneumonia),
}

print("Class weights:", class_weight)

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=[early_stop],
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

y_true = np.concatenate([y.numpy() for x, y in test_ds])
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]))

In [ ]:
data_aug = models.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

model_aug = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_aug,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model_aug.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stop2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history_aug = model_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=[early_stop2],
)

In [ ]:
test_loss2, test_acc2 = model_aug.evaluate(test_ds)
print("Test accuracy (augmented):", test_acc2)

y_prob2 = model_aug.predict(test_ds).ravel()
y_pred2 = (y_prob2 > 0.5).astype(int)

print(confusion_matrix(y_true, y_pred2))
print(classification_report(y_true, y_pred2, target_names=["NORMAL", "PNEUMONIA"]))

In [ ]:
import numpy as np

print("Evaluating Data Augmented Model (Model 2) on Test Dataset...")
loss_aug, acc_aug = model_aug.evaluate(test_ds)
print(f"Test Accuracy: {acc_aug * 100:.2f}%")

predictions_aug = model_aug.predict(test_ds)
y_pred_aug = np.where(predictions_aug > 0.5, 1, 0).flatten()

cm_aug = confusion_matrix(y_true, y_pred_aug)
print("\nConfusion Matrix:")
print(cm_aug)

report_aug = classification_report(y_true, y_pred_aug, target_names=['NORMAL', 'PNEUMONIA'])
print("\nClassification Report:")
print(report_aug)


In [ ]:
import matplotlib.pyplot as plt

metrics = ['Accuracy', 'Normal Recall', 'Pneumonia Recall']
model1_scores = [0.761, 0.38, 0.99]
model2_scores = [0.8333, 0.59, 0.98]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
rects1 = ax.bar(x - width/2, model1_scores, width, label='Model 1 (No Augmentation)', color='skyblue')
rects2 = ax.bar(x + width/2, model2_scores, width, label='Model 2 (With Augmentation)', color='salmon')

ax.set_ylabel('Scores')
ax.set_title('Comparison between Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_mobilenet = models.Model(inputs, outputs)

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.summary()


In [ ]:
early_stop_mobilenet = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Starting MobileNetV2 Training...")
history_mobilenet = model_mobilenet.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    class_weight={0: 1.944, 1: 0.673},
    callbacks=[early_stop_mobilenet]
)


In [ ]:
import numpy as np

print("Evaluating MobileNetV2 Model on Test Dataset...")
loss_mobilenet, acc_mobilenet = model_mobilenet.evaluate(test_ds)
print(f"Test Accuracy: {acc_mobilenet * 100:.2f}%")

predictions_mobilenet = model_mobilenet.predict(test_ds)
y_pred_mobilenet = np.where(predictions_mobilenet > 0.5, 1, 0).flatten()

cm_mobilenet = confusion_matrix(y_true, y_pred_mobilenet)
print("\nConfusion Matrix:")
print(cm_mobilenet)

report_mobilenet = classification_report(y_true, y_pred_mobilenet, target_names=['NORMAL', 'PNEUMONIA'])
print("\nClassification Report:")
print(report_mobilenet)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

data_augmentation_resnet = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

base_model_resnet = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model_resnet.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation_resnet(inputs)
x = tf.keras.applications.resnet50.preprocess_input(x)
x = base_model_resnet(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_resnet = models.Model(inputs, outputs)

model_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_resnet.summary()


In [ ]:
early_stop_resnet = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Starting ResNet50 Training...")
history_resnet = model_resnet.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    class_weight={0: 1.944, 1: 0.673},
    callbacks=[early_stop_resnet]
)


In [ ]:
import numpy as np

print("Evaluating ResNet50 Model on Test Dataset...")
loss_resnet, acc_resnet = model_resnet.evaluate(test_ds)
print(f"Test Accuracy: {acc_resnet * 100:.2f}%")

predictions_resnet = model_resnet.predict(test_ds)
y_pred_resnet = np.where(predictions_resnet > 0.5, 1, 0).flatten()

cm_resnet = confusion_matrix(y_true, y_pred_resnet)
print("\nConfusion Matrix:")
print(cm_resnet)

report_resnet = classification_report(y_true, y_pred_resnet, target_names=['NORMAL', 'PNEUMONIA'])
print("\nClassification Report:")
print(report_resnet)


In [ ]:
import pandas as pd

data = {
    'Model Name': [
        'Custom CNN (No Aug)',
        'Custom CNN (With Aug)',
        'MobileNetV2 (Freeze)',
        'ResNet50 (Freeze)'
    ],
    'Test Accuracy (%)': [76.10, 83.33, 80.61, 79.17],
    'Normal Precision': [0.96, 0.95, 0.97, 0.91],
    'Normal Recall': [0.38, 0.59, 0.50, 0.49],
    'Pneumonia Precision': [0.73, 0.80, 0.77, 0.76],
    'Pneumonia Recall': [0.99, 0.98, 0.99, 0.97]
}

df_comparison = pd.DataFrame(data)

print("=== Comparative Analysis of Deep Learning Models ===")
display(df_comparison)
import pandas as pd

data = {
    'Model Name': [
        'Custom CNN (No Aug)',
        'Custom CNN (With Aug)',
        'MobileNetV2 (Freeze)',
        'ResNet50 (Freeze)'
    ],
    'Test Accuracy (%)': [76.10, 83.33, 80.61, 79.17],
    'Normal Precision': [0.96, 0.95, 0.97, 0.91],
    'Normal Recall': [0.38, 0.59, 0.50, 0.49],
    'Pneumonia Precision': [0.73, 0.80, 0.77, 0.76],
    'Pneumonia Recall': [0.99, 0.98, 0.99, 0.97]
}

df_comparison = pd.DataFrame(data)

print("=== Comparative Analysis of Deep Learning Models ===")
display(df_comparison)


In [ ]:
from google.colab import files

df_comparison.to_csv('pneumonia_models_comparison.csv', index=False)

print("Saving table and preparing for download...")
files.download('pneumonia_models_comparison.csv')
from google.colab import files

df_comparison.to_csv('pneumonia_models_comparison.csv', index=False)

print("Saving table and preparing for download...")
files.download('pneumonia_models_comparison.csv')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Custom CNN (No Augmentation)
axes[0, 0].plot(history.history['accuracy'], label='Train Acc', color='blue')
axes[0, 0].plot(history.history['val_accuracy'], label='Val Acc', color='orange')
axes[0, 0].set_title('Custom CNN (No Aug) - Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2. Custom CNN (With Augmentation)
axes[0, 1].plot(history_aug.history['accuracy'], label='Train Acc', color='green')
axes[0, 1].plot(history_aug.history['val_accuracy'], label='Val Acc', color='red')
axes[0, 1].set_title('Custom CNN (With Aug) - Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# 3. MobileNetV2
axes[1, 0].plot(history_mobilenet.history['accuracy'], label='Train Acc', color='purple')
axes[1, 0].plot(history_mobilenet.history['val_accuracy'], label='Val Acc', color='brown')
axes[1, 0].set_title('MobileNetV2 - Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True)

# 4. ResNet50
axes[1, 1].plot(history_resnet.history['accuracy'], label='Train Acc', color='cyan')
axes[1, 1].plot(history_resnet.history['val_accuracy'], label='Val Acc', color='magenta')
axes[1, 1].set_title('ResNet50 - Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.suptitle('Training and Validation Accuracy Curves for All Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

cm1 = [[89, 145], [4, 386]]
cm2 = [[138, 96], [8, 382]]
cm3 = [[117, 117], [4, 386]]
cm4 = [[115, 119], [11, 379]]

labels = ['NORMAL', 'PNEUMONIA']

# 1. Custom CNN (No Aug)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], xticklabels=labels, yticklabels=labels)
axes[0, 0].set_title('Custom CNN (No Aug)')
axes[0, 0].set_ylabel('Actual')
axes[0, 0].set_xlabel('Predicted')

# 2. Custom CNN (With Aug)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=axes[0, 1], xticklabels=labels, yticklabels=labels)
axes[0, 1].set_title('Custom CNN (With Aug)')
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_xlabel('Predicted')

# 3. MobileNetV2
sns.heatmap(cm3, annot=True, fmt='d', cmap='Purples', ax=axes[1, 0], xticklabels=labels, yticklabels=labels)
axes[1, 0].set_title('MobileNetV2 (Freeze)')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_xlabel('Predicted')

# 4. ResNet50
sns.heatmap(cm4, annot=True, fmt='d', cmap='Oranges', ax=axes[1, 1], xticklabels=labels, yticklabels=labels)
axes[1, 1].set_title('ResNet50 (Freeze)')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices Heatmap for All Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

print("Grad-CAM function successfully created!")
!$0

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

def show_images(cls, n=4):
    folder = os.path.join(base, "train", cls)
    files = random.sample(os.listdir(folder), n)
    plt.figure(figsize=(12, 3))
    for i, name in enumerate(files):
        img = Image.open(os.path.join(folder, name))
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(cls)
        plt.axis("off")
    plt.show()

show_images("NORMAL")
show_images("PNEUMONIA")

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.cm as cm
import matplotlib.pyplot as plt

# 1. New Modified Grad-CAM function that can access base model layers
def make_mobilenet_gradcam(img_array, top_model, base_layer_name, last_conv_layer_name, pred_index=None):
    base_model = top_model.get_layer(base_layer_name)

    # Internal model mapping from base input to target conv layer and final predictions
    grad_model = tf.keras.models.Model(
        inputs=[base_model.inputs],
        outputs=[base_model.get_layer(last_conv_layer_name).output, top_model.output]
    )

    with tf.GradientTape() as tape:
        # Preprocess input inside MobileNetV2 range before passing
        processed_input = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
        last_conv_layer_output, preds = grad_model(processed_input)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

# 2. Dynamically find the layer name exactly as you did
mobilenet_base_name = 'mobilenetv2_1.00_224'
mobilenet_base_layer = model_mobilenet.get_layer(mobilenet_base_name)

last_conv_layer_name = None
for layer in reversed(mobilenet_base_layer.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

print(f"Detected last convolutional layer: {last_conv_layer_name}")

# 3. Extract one image batch from test dataset
for images, labels in test_ds.take(1):
    img_array = images[0:1].numpy()
    true_label = labels[0].numpy()
    break

# 4. Generate heatmap matrix using the new safe function
heatmap = make_mobilenet_gradcam(img_array, model_mobilenet, mobilenet_base_name, last_conv_layer_name)

# 5. Process original image and resize heatmap using your exact dimensions
img = img_array[0]
img = np.uint8(255 * img)

heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
heatmap_resized = np.uint8(255 * heatmap_resized)

# 6. Apply Jet color map and blend images
heatmap_cmap = cm.get_cmap("jet")
heatmap_colors = heatmap_cmap(heatmap_resized)[:, :, :3]
heatmap_colors = np.uint8(255 * heatmap_colors)

superimposed_img = heatmap_colors * 0.4 + img * 0.6
superimposed_img = np.uint8(superimposed_img)

# 7. Plot the original image, heatmap, and superimposed image side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img)
axes[0].set_title(f"Original X-ray (Class: {true_label})")
axes[0].axis("off")

axes[1].imshow(heatmap, cmap="jet")
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis("off")

axes[2].imshow(superimposed_img)
axes[2].set_title("Superimposed Grad-CAM")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

literature_data = {
    'Author & Year': [
        'Mooney et al. (2018)',
        'Rahman et al. (2020)',
        'Sharma et al. (2022)',
        'Fahim et al. (2026) - Ours'
    ],
    'Methodology / Models Used': [
        'Baseline CNN Model',
        'Modified ResNet50 Transfer Learning',
        'VGG16 with Data Augmentation',
        'Custom CNN (Aug) vs MobileNetV2 & ResNet50'
    ],
    'Reported Accuracy (%)': [78.40, 89.20, 85.60, 83.33],
    'Key Contribution': [
        'Provided the original dataset on Kaggle.',
        'Improved pneumonia detection using deeper networks.',
        'Addressed over-fitting issues using dropouts.',
        'Demonstrated that custom tailored light CNN can outperform frozen deep transfer networks.'
    ]
}

df_literature = pd.DataFrame(literature_data)

print("=== Literature Review and Comparative Reference Table ===")
display(df_literature)
import pandas as pd

literature_data = {
    'Author & Year': [
        'Mooney et al. (2018)',
        'Rahman et al. (2020)',
        'Sharma et al. (2022)',
        'Fahim et al. (2026) - Ours'
    ],
    'Methodology / Models Used': [
        'Baseline CNN Model',
        'Modified ResNet50 Transfer Learning',
        'VGG16 with Data Augmentation',
        'Custom CNN (Aug) vs MobileNetV2 & ResNet50'
    ],
    'Reported Accuracy (%)': [78.40, 89.20, 85.60, 83.33],
    'Key Contribution': [
        'Provided the original dataset on Kaggle.',
        'Improved pneumonia detection using deeper networks.',
        'Addressed over-fitting issues using dropouts.',
        'Demonstrated that custom tailored light CNN can outperform frozen deep transfer networks.'
    ]
}

df_literature = pd.DataFrame(literature_data)

print("=== Literature Review and Comparative Reference Table ===")
display(df_literature)


In [ ]:
import graphviz

dot = graphviz.Digraph(comment='Methodology Pipeline', format='png')

dot.attr(rankdir='TB', size='10,12', dpi='300')
dot.attr('node', shape='box', style='filled, rounded', color='black', fillcolor='#F5F5F5', fontname='Arial', fontsize='11')
dot.attr('edge', color='black', arrowhead='vee', fontname='Arial', fontsize='10')

dot.node('A', 'Input Dataset\n(Kaggle Chest X-ray Corpus\n5,856 Images)')
dot.node('B', 'Data Stratification & Partitioning\n(80:20 Training/Validation Restructuring)')
dot.node('C', 'Training Subset\n(4,173 Images)', fillcolor='#EAEAEA')
dot.node('D', 'Validation Subset\n(1,043 Images)', fillcolor='#EAEAEA')
dot.node('E', 'Isolated Test Cohort\n(624 Images - shuffle=False)', fillcolor='#DCDCDC')
dot.node('F', 'Image Preprocessing\n(Resize 224x224 | RGB Mode | Batch Size: 32)')
dot.node('G', 'Class Imbalance Mitigation\n(Class Weights -> Normal: 1.944, Pneumonia: 0.673)')
dot.node('H', 'Sequential Data Augmentation\n(Random Rotation 0.05 | Zoom 0.1 | Contrast 0.1)')

dot.node('I1', 'Architecture A\nCustom CNN Framework\n(11M Trainable Params)')
dot.node('I2', 'Architecture B\nMobileNetV2 Paradigm\n(Frozen Base Extractor)')
dot.node('I3', 'Architecture C\nResNet50 Paradigm\n(Frozen Residual Extractor)')

dot.node('J', 'Optimization & Convergence Controls\n(Adam Optimizer lr=0.001 | EarlyStopping patience=3)')
dot.node('K', 'Diagnostic Node\n(Sigmoid Activation -> Binary Inference)')
dot.node('L', 'Rigorous Statistical Evaluation\n(Accuracy, Precision, Recall, Confusion Matrix)')

dot.edge('A', 'B')
dot.edge('B', 'C')
dot.edge('B', 'D')
dot.edge('B', 'E')
dot.edge('C', 'F')
dot.edge('D', 'F')
dot.edge('F', 'G')
dot.edge('G', 'H')
dot.edge('H', 'I1')
dot.edge('H', 'I2')
dot.edge('H', 'I3')
dot.edge('I1', 'J')
dot.edge('I2', 'J')
dot.edge('I3', 'J')
dot.edge('J', 'K')
dot.edge('K', 'L')
dot.edge('E', 'L')

dot.render('methodology_architecture_flowchart', view=False)
dot


```mermaid
graph TD
    A[Input Dataset: Kaggle Chest X-ray<br>5,856 Images] --> B(Data Stratification & Partitioning)
    B --> C[Training Subset<br>4,173 Images]
    B --> D[Validation Subset<br>1,043 Images]
    B --> E[Isolated Test Cohort<br>624 Images]
    
    C --> F[Image Preprocessing<br>Resize 224x224 | RGB]
    D --> F
    F --> G[Imbalance Mitigation<br>Class Weights Applied]
    G --> H[Data Augmentation<br>Rotation | Zoom | Contrast]
    
    H --> I1[Custom CNN<br>11M Params]
    H --> I2[MobileNetV2<br>Frozen Base]
    H --> I3[ResNet50<br>Frozen Base]
    
    I1 --> J[Adam Optimizer & val_loss Validation]
    I2 --> J
    I3 --> J
    J --> K[Diagnostic Node<br>Sigmoid Binary Inference]
    K --> L[Statistical Evaluation<br>Accuracy, Precision, Recall, CM]
    E --> L
```
S

In [ ]:
import cv2
import numpy as np
import IPython
from IPython.display import display, Image
from google.colab import files

# Professional configuration for the methodology architecture diagram
mermaid_code = """
graph TD
    A[Input Dataset: Kaggle Chest X-ray<br>5,856 Images] --> B(Data Stratification & Partitioning)
    B --> C[Training Subset<br>4,173 Images]
    B --> D[Validation Subset<br>1,043 Images]
    B --> E[Isolated Test Cohort<br>624 Images]

    C --> F[Image Preprocessing<br>Resize 224x224 | RGB]
    D --> F
    F --> G[Imbalance Mitigation<br>Class Weights Applied]
    G --> H[Data Augmentation<br>Rotation | Zoom | Contrast]

    H --> I1[Custom CNN<br>11M Params]
    H --> I2[MobileNetV2<br>Frozen Base]
    H --> I3[ResNet50<br>Frozen Base]

    I1 --> J[Adam Optimizer & val_loss Validation]
    I2 --> J
    I3 --> J
    J --> K[Diagnostic Node<br>Sigmoid Binary Inference]
    K --> L[Statistical Evaluation<br>Accuracy, Precision, Recall, CM]
    E --> L
"""

# Fetch the high-resolution image from the official Mermaid engine
url = "https://mermaid.ink" + mermaid_code.strip().replace('\n', '%0A').replace(' ', '%20')

print("Generating methodology flowchart...")
display(Image(url=url))

# Automatically download the diagram file to your computer
import urllib.request
urllib.request.urlretrieve(url, "methodology_flowchart.png")
print("\nSuccess! Diagram saved as 'methodology_flowchart.png'")
files.download("methodology_flowchart.png")


In [ ]:
import graphviz
from google.colab import files

# 1. Initialize a professional directional diagram using local Graphviz configuration
dot = graphviz.Digraph(comment='Methodology Pipeline Structure', format='png')

# 2. Set globally accepted academic styling parameters
dot.attr(rankdir='TB', size='10,12', dpi='300')
dot.attr('node', shape='box', style='filled, rounded', color='black', fillcolor='#F8F9FA', fontname='Arial', fontsize='10')
dot.attr('edge', color='black', arrowhead='vee', arrowsize='0.8', fontname='Arial', fontsize='9')

# 3. Define the framework stages with precise project metrics
dot.node('A', 'Input Dataset\n(Kaggle Chest X-ray Corpus\n5,856 Initial Images)')
dot.node('B', 'Data Stratification & Partitioning\n(80:20 Dynamic Split Strategy)')
dot.node('C', 'Training Cohort\n(4,173 Images)', fillcolor='#F1F3F4')
dot.node('D', 'Validation Cohort\n(1,043 Images)', fillcolor='#F1F3F4')
dot.node('E', 'Isolated Test Subset\n(624 Images | shuffle=False)', fillcolor='#E8EAED')
dot.node('F', 'Image Normalization Pipeline\n(Resize: 224x224 | Color Mode: RGB | Batch: 32)')
dot.node('G', 'Class Imbalance Penalization\n(Class Weights -> Normal: 1.944 | Pneumonia: 0.673)')
dot.node('H', 'Sequential Data Augmentation\n(Random Rotation: 0.05 | Zoom: 0.1 | Contrast: 0.1)')

# Computational architecture nodes
dot.node('I1', 'Framework A:\nCustom CNN\n(11M Parameters)')
dot.node('I2', 'Framework B:\nMobileNetV2\n(Frozen Base)')
dot.node('I3', 'Framework C:\nResNet50\n(Frozen Base)')

dot.node('J', 'Optimization & Convergence Layer\n(Adam Optimizer lr=0.001 | EarlyStopping patience=3)')
dot.node('K', 'Structural Diagnostic Node\n(Sigmoid Activation Function)')
dot.node('L', 'Rigorous Statistical Evaluation\n(Accuracy, Precision, Recall, Confusion Matrix)')

# 4. Map out the architectural connectivity paths
dot.edge('A', 'B')
dot.edge('B', 'C')
dot.edge('B', 'D')
dot.edge('B', 'E')
dot.edge('C', 'F')
dot.edge('D', 'F')
dot.edge('F', 'G')
dot.edge('G', 'H')
dot.edge('H', 'I1')
dot.edge('H', 'I2')
dot.edge('H', 'I3')
dot.edge('I1', 'J')
dot.edge('I2', 'J')
dot.edge('I3', 'J')
dot.edge('J', 'K')
dot.edge('K', 'L')
dot.edge('E', 'L')

# 5. Render to local disk and trigger browser download sequence
output_filename = 'methodology_pipeline_flowchart'
dot.render(output_filename, view=False)
print("Flowchart image successfully rendered on local disk!")

# Trigger download to user laptop
files.download(f"{output_filename}.png")


In [ ]:
import graphviz
from google.colab import files

# Initialize a premium directional graph for Scopus/IEEE journal publications
dot = graphviz.Digraph(comment='Premium Methodology Pipeline', format='png')

# Set high-resolution output and clean aesthetic attributes
dot.attr(rankdir='TB', size='12,15', dpi='300')
dot.attr('graph', bgcolor='#FFFFFF', fontname='Helvetica')

# 1. High-Quality Color Palette Coding for Distinct Pipeline Steps
# Light Blue: Data Phase | Teal: Processing | Orange: Models | Green: Evaluation
dot.attr('node', shape='box', style='filled, rounded', fontname='Helvetica', fontsize='10', penwidth='1.5')
dot.attr('edge', color='#2C3E50', arrowhead='vee', arrowsize='0.9', penwidth='1.2')

# Define Data Ingestion & Splitting Block (Light Blue theme)
dot.node('A', 'Input Dataset\n(Kaggle Chest X-ray Corpus\n5,856 Total Images)', fillcolor='#EBF5FB', color='#2E86C1', fontcolor='#1B4F72')
dot.node('B', 'Data Stratification & Partitioning\n(80:20 Dynamic Split Matrix)', fillcolor='#EBF5FB', color='#2E86C1', fontcolor='#1B4F72')
dot.node('C', 'Training Cohort\n(4,173 Images)', fillcolor='#EBF5FB', color='#2E86C1', fontcolor='#1B4F72')
dot.node('D', 'Validation Cohort\n(1,043 Images)', fillcolor='#EBF5FB', color='#2E86C1', fontcolor='#1B4F72')
dot.node('E', 'Isolated Test Cohort\n(624 Images | shuffle=False)', fillcolor='#FDEDEC', color='#CB4335', fontcolor='#78281F')

# Define Image Preprocessing and Engineering Block (Teal theme)
dot.node('F', 'Image Normalization Pipeline\n(Resize: 224x224 | Color Mode: RGB | Batch: 32)', fillcolor='#E8F8F5', color='#117A65', fontcolor='#0E6251')
dot.node('G', 'Class Imbalance Penalization\n(Class Weights -> Normal: 1.944 | Pneumonia: 0.673)', fillcolor='#E8F8F5', color='#117A65', fontcolor='#0E6251')
dot.node('H', 'Sequential Data Augmentation\n(Random Rotation: 0.05 | Zoom: 0.1 | Contrast: 0.1)', fillcolor='#E8F8F5', color='#117A65', fontcolor='#0E6251')

# Define Deep Learning Framework Block (Orange theme)
dot.node('I1', 'Framework A:\nCustom CNN\n(11M Parameters)', fillcolor='#FEF9E7', color='#D4AC0D', fontcolor='#7D6608')
dot.node('I2', 'Framework B:\nMobileNetV2\n(Frozen Base Extractor)', fillcolor='#FEF9E7', color='#D4AC0D', fontcolor='#7D6608')
dot.node('I3', 'Framework C:\nResNet50\n(Frozen Residual Extractor)', fillcolor='#FEF9E7', color='#D4AC0D', fontcolor='#7D6608')

# Define Optimization and Inference Block (Dark Blue/Gray theme)
dot.node('J', 'Optimization & Convergence Controls\n(Adam Optimizer lr=0.001 | EarlyStopping patience=3)', fillcolor='#F2F3F4', color='#7F8C8D', fontcolor='#2C3E50')
dot.node('K', 'Structural Diagnostic Node\n(Sigmoid Activation Function)', fillcolor='#F2F3F4', color='#7F8C8D', fontcolor='#2C3E50')

# Define Final Statistical Evaluation Block (Green theme)
dot.node('L', 'Rigorous Statistical Evaluation\n(Accuracy, Precision, Recall, Confusion Matrix)', fillcolor='#EAF2F8', color='#2471A3', fontcolor='#1A5276')

# 2. Re-map the structural pipeline flow
dot.edge('A', 'B')
dot.edge('B', 'C')
dot.edge('B', 'D')
dot.edge('B', 'E')
dot.edge('C', 'F')
dot.edge('D', 'F')
dot.edge('F', 'G')
dot.edge('G', 'H')
dot.edge('H', 'I1')
dot.edge('H', 'I2')
dot.edge('H', 'I3')
dot.edge('I1', 'J')
dot.edge('I2', 'J')
dot.edge('I3', 'J')
dot.edge('J', 'K')
dot.edge('K', 'L')
dot.edge('E', 'L')

# Render the dynamic vector chart locally
output_filename = 'premium_methodology_flowchart'
dot.render(output_filename, view=False)
print("Journal-ready flowchart successfully generated!")

# Trigger automatic browser download
files.download(f"{output_filename}.png")
